# ⚠️ Pipeline Reset

Notebook này xóa sạch toàn bộ để chạy lại pipeline từ đầu.

**Những gì sẽ bị xóa:**
- Tất cả Iceberg tables (Bronze, Silver, Gold) — catalog metadata
- Tất cả data files & metadata files trên MinIO (`bronze/`, `silver/`, `gold/`)
- Tất cả Spark Streaming checkpoints trên MinIO
- Kafka topic `raw_yelp_users` (xóa + tạo lại)

**Những gì KHÔNG bị xóa:**
- File Yelp gốc tại `/home/jovyan/data/yelp/` — an toàn
- Cấu hình Docker Compose

---
⚠️ **Chỉ chạy khi muốn reset hoàn toàn. Không thể undo.**

In [1]:
# Xác nhận trước khi reset
confirm = input("Nhập 'RESET' để xác nhận xóa toàn bộ data: ")
if confirm.strip() != "RESET":
    raise SystemExit("❌ Hủy reset — không có gì bị thay đổi")
print("✅ Xác nhận nhận được — bắt đầu reset...")

✅ Xác nhận nhận được — bắt đầu reset...


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Pipeline_Reset") \
    .config("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog") \
    .config("spark.sql.catalog.nessie.uri", "http://nessie:19120/api/v1") \
    .config("spark.sql.catalog.nessie.ref", "main") \
    .config("spark.sql.catalog.nessie.warehouse", "s3a://warehouse/") \
    .config("spark.sql.catalog.nessie.s3.endpoint", "http://minio:9000") \
    .config("spark.sql.catalog.nessie.io-impl", "org.apache.iceberg.io.ResolvingFileIO") \
    .config("spark.sql.catalog.nessie.s3.path-style-access", "true") \
    .config("spark.sql.catalog.nessie.s3.access-key-id", "admin") \
    .config("spark.sql.catalog.nessie.s3.secret-access-key", "password") \
    .config("spark.sql.defaultCatalog", "nessie") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .getOrCreate()

hc = spark.sparkContext._jsc.hadoopConfiguration()
hc.set("fs.s3a.endpoint",               "http://minio:9000")
hc.set("fs.s3a.access.key",             "admin")
hc.set("fs.s3a.secret.key",             "password")
hc.set("fs.s3a.path.style.access",      "true")
hc.set("fs.s3a.connection.ssl.enabled", "false")
hc.set("fs.s3a.impl",                   "org.apache.hadoop.fs.s3a.S3AFileSystem")
hc.set("fs.s3a.aws.credentials.provider",
       "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")

spark.sparkContext.setLogLevel("ERROR")
print("✅ SparkSession ready")

✅ SparkSession ready


26/06/16 01:12:39 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [3]:
# ── KHỞI TẠO boto3 CLIENT (dùng chung cho các bước sau) ──────────
import boto3
from botocore.client import Config

s3 = boto3.client(
    "s3",
    endpoint_url="http://minio:9000",
    aws_access_key_id="admin",
    aws_secret_access_key="password",
    config=Config(signature_version="s3v4"),
)

def delete_s3_prefix(bucket, prefix):
    """Xóa toàn bộ objects có prefix trong bucket. Trả về số objects đã xóa."""
    paginator = s3.get_paginator("list_objects_v2")
    deleted = 0
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        objects = page.get("Contents", [])
        if objects:
            s3.delete_objects(
                Bucket=bucket,
                Delete={"Objects": [{"Key": o["Key"]} for o in objects]}
            )
            deleted += len(objects)
    return deleted

print("✅ boto3 S3 client ready")

✅ boto3 S3 client ready


In [4]:
# ── BƯỚC 1: Stop tất cả Spark Streaming queries ──────────────────
print("BƯỚC 1: Stop Streaming Queries")
print("-" * 40)

active = spark.streams.active
if not active:
    print("  Không có stream nào đang chạy")
else:
    for q in active:
        q.stop()
        print(f"  ⏹  Stopped: {q.name}")

print("  ✅ Done")

BƯỚC 1: Stop Streaming Queries
----------------------------------------
  Không có stream nào đang chạy
  ✅ Done


In [5]:
# ── BƯỚC 2: Drop Iceberg tables khỏi Nessie catalog ──────────────
print("BƯỚC 2: Drop Iceberg Tables (Nessie catalog)")
print("-" * 40)

tables = [
    "nessie.gold.businesses_flat",
    "nessie.silver.yelp_users_scd2",
    "nessie.bronze.yelp_users",
    # Olist tables (nếu còn sót)
    "nessie.silver.customers_scd2",
    "nessie.bronze.customers",
]

for t in tables:
    try:
        spark.sql(f"DROP TABLE IF EXISTS {t}")
        print(f"  🗑  Dropped : {t}")
    except Exception as e:
        print(f"  Skip       : {t} ({e})")

# Drop namespaces
for ns in ["nessie.gold", "nessie.silver", "nessie.bronze"]:
    try:
        spark.sql(f"DROP NAMESPACE IF EXISTS {ns}")
        print(f"  🗑  Namespace dropped: {ns}")
    except Exception as e:
        print(f"  Skip namespace: {ns} ({e})")

print("  ✅ Done")

BƯỚC 2: Drop Iceberg Tables (Nessie catalog)
----------------------------------------
  🗑  Dropped : nessie.gold.businesses_flat
  🗑  Dropped : nessie.silver.yelp_users_scd2
  🗑  Dropped : nessie.bronze.yelp_users
  🗑  Dropped : nessie.silver.customers_scd2
  🗑  Dropped : nessie.bronze.customers
  🗑  Namespace dropped: nessie.gold
  🗑  Namespace dropped: nessie.silver
  🗑  Namespace dropped: nessie.bronze
  ✅ Done


In [6]:
# ── BƯỚC 3: Xóa data files & metadata files trên MinIO ───────────
#
# DROP TABLE chỉ xóa catalog entry, KHÔNG xóa .parquet/.avro/.json
# trên S3 → phải xóa thủ công để MinIO thực sự sạch.
#
print("BƯỚC 3: Xóa data/metadata files trên MinIO")
print("-" * 40)

data_prefixes = [
    "bronze/",
    "silver/",
    "gold/",
]

for prefix in data_prefixes:
    n = delete_s3_prefix("warehouse", prefix)
    if n > 0:
        print(f"  🗑  Deleted: warehouse/{prefix} ({n} objects)")
    else:
        print(f"  Skip (empty): warehouse/{prefix}")

print("  ✅ Done")

BƯỚC 3: Xóa data/metadata files trên MinIO
----------------------------------------
  🗑  Deleted: warehouse/bronze/ (129 objects)
  🗑  Deleted: warehouse/silver/ (265 objects)
  Skip (empty): warehouse/gold/
  ✅ Done


In [7]:
# ── BƯỚC 4: Xóa Spark Streaming Checkpoints trên MinIO ───────────
print("BƯỚC 4: Xóa Checkpoints trên MinIO")
print("-" * 40)

checkpoints = [
    "checkpoints/bronze_yelp_users",
    "checkpoints/silver_yelp_users_scd2",
    # Olist checkpoints (nếu còn sót)
    "checkpoints/bronze_customers",
    "checkpoints/silver_customers_scd2",
]

for prefix in checkpoints:
    n = delete_s3_prefix("warehouse", prefix)
    if n > 0:
        print(f"  🗑  Deleted: warehouse/{prefix} ({n} objects)")
    else:
        print(f"  Skip (empty): warehouse/{prefix}")

print("  ✅ Done")

BƯỚC 4: Xóa Checkpoints trên MinIO
----------------------------------------
  🗑  Deleted: warehouse/checkpoints/bronze_yelp_users (46 objects)
  🗑  Deleted: warehouse/checkpoints/silver_yelp_users_scd2 (34 objects)
  Skip (empty): warehouse/checkpoints/bronze_customers
  Skip (empty): warehouse/checkpoints/silver_customers_scd2
  ✅ Done


In [8]:
# ── BƯỚC 5: Reset Kafka Topic ─────────────────────────────────────
from kafka.admin import KafkaAdminClient, NewTopic
from kafka.errors import UnknownTopicOrPartitionError
import time

TOPIC = "raw_yelp_users"

print("BƯỚC 5: Reset Kafka Topic")
print("-" * 40)

admin = KafkaAdminClient(
    bootstrap_servers="kafka:9092",
    client_id="pipeline_reset"
)

# Xóa topic nếu tồn tại
existing_topics = admin.list_topics()
if TOPIC in existing_topics:
    admin.delete_topics([TOPIC])
    print(f"  🗑  Deleted topic: {TOPIC}")
    time.sleep(5)  # Chờ Kafka xử lý xong deletion
else:
    print(f"  Skip: topic '{TOPIC}' chưa tồn tại")

# Tạo lại topic mới
new_topic = NewTopic(
    name=TOPIC,
    num_partitions=3,
    replication_factor=1
)
admin.create_topics([new_topic])
print(f"  ✅ Created topic: {TOPIC} (3 partitions)")

admin.close()
print("  ✅ Done")

BƯỚC 5: Reset Kafka Topic
----------------------------------------
  🗑  Deleted topic: raw_yelp_users
  ✅ Created topic: raw_yelp_users (3 partitions)
  ✅ Done


In [9]:
# ── BƯỚC 6: Verify toàn bộ đã sạch ──────────────────────────────
print("BƯỚC 6: Verification")
print("-" * 40)

all_ok = True

# 6a. Check Nessie catalog — tables đã drop chưa
for ns in ["nessie.bronze", "nessie.silver", "nessie.gold"]:
    try:
        remaining = spark.sql(f"SHOW TABLES IN {ns}").count()
        if remaining == 0:
            print(f"  ✅ Catalog {ns}: sạch")
        else:
            print(f"  ⚠️  Catalog {ns}: còn {remaining} table chưa bị drop")
            all_ok = False
    except Exception:
        # Namespace không tồn tại = đã drop hoàn toàn = OK
        print(f"  ✅ Catalog {ns}: namespace không tồn tại (đã xóa hoàn toàn)")

# 6b. Check MinIO — data folders đã sạch chưa
for prefix in ["bronze/", "silver/", "gold/"]:
    result = s3.list_objects_v2(Bucket="warehouse", Prefix=prefix, MaxKeys=1)
    count = result.get("KeyCount", 0)
    if count == 0:
        print(f"  ✅ MinIO warehouse/{prefix}: sạch")
    else:
        print(f"  ⚠️  MinIO warehouse/{prefix}: còn objects")
        all_ok = False

# 6c. Check checkpoints
cp_result = s3.list_objects_v2(Bucket="warehouse", Prefix="checkpoints/", MaxKeys=1)
cp_count = cp_result.get("KeyCount", 0)
if cp_count == 0:
    print("  ✅ Checkpoints: sạch")
else:
    print(f"  ⚠️  Còn checkpoint objects")
    all_ok = False

# 6d. Check streams
active_streams = len(spark.streams.active)
if active_streams == 0:
    print("  ✅ Streams: không có stream nào đang chạy")
else:
    print(f"  ⚠️  Còn {active_streams} streams đang chạy")
    all_ok = False

print()
if all_ok:
    print("╔══════════════════════════════════════════╗")
    print("║  ✅ RESET HOÀN TẤT — Sẵn sàng chạy lại  ║")
    print("╚══════════════════════════════════════════╝")
    print()
    print("Thứ tự chạy lại:")
    print("  1. spark_bronze_yelp.ipynb       (chờ log stream đang chạy)")
    print("  2. kafka_producer_yelp.ipynb      (Pass 1 — full load)")
    print("  3. spark_silver_yelp_scd2.ipynb   (start khi Bronze đang consume)")
    print("  4. kafka_producer_yelp.ipynb      (Pass 2 khi Bronze ~2M rows)")
    print("  5. kafka_producer_yelp.ipynb      (Pass 3)")
    print("  6. spark_gold_businesses.ipynb    (batch, sau Silver có expired > 0)")
    print("  7. system_benchmark.ipynb         (chạy sau cùng)")
else:
    print("⚠️  Một số bước chưa sạch — kiểm tra warning ở trên")

BƯỚC 6: Verification
----------------------------------------
  ✅ Catalog nessie.bronze: sạch
  ✅ Catalog nessie.silver: sạch
  ✅ Catalog nessie.gold: sạch
  ✅ MinIO warehouse/bronze/: sạch
  ✅ MinIO warehouse/silver/: sạch
  ✅ MinIO warehouse/gold/: sạch
  ✅ Checkpoints: sạch
  ✅ Streams: không có stream nào đang chạy

╔══════════════════════════════════════════╗
║  ✅ RESET HOÀN TẤT — Sẵn sàng chạy lại  ║
╚══════════════════════════════════════════╝

Thứ tự chạy lại:
  1. spark_bronze_yelp.ipynb       (chờ log stream đang chạy)
  2. kafka_producer_yelp.ipynb      (Pass 1 — full load)
  3. spark_silver_yelp_scd2.ipynb   (start khi Bronze đang consume)
  4. kafka_producer_yelp.ipynb      (Pass 2 khi Bronze ~2M rows)
  5. kafka_producer_yelp.ipynb      (Pass 3)
  6. spark_gold_businesses.ipynb    (batch, sau Silver có expired > 0)
  7. system_benchmark.ipynb         (chạy sau cùng)
